# Merge and Validate LLM Result Datasets
This notebook merges specified CSV pairs into full `test`, `val`, and `train` datasets, then performs validation checks: error counts, duplicate responses per prompt, prompt counts, language/variant coverage, and produces deduplicated outputs (one response per prompt).

In [7]:
import pandas as pd
from pathlib import Path
from typing import Optional, List, Dict

# Helpers to detect likely column names
def find_column(df: pd.DataFrame, keywords: List[str]) -> Optional[str]:
    for k in keywords:
        for col in df.columns:
            if k.lower() in col.lower():
                return col
    return None

def safe_read_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f'Failed to read {path}: {e}')
        return pd.DataFrame()

In [8]:
# Locate project root (search upward for pyproject.toml, README.md, or .git)
project_root = Path.cwd()
for p in [Path('.'), Path.cwd()]:
    cur = p.resolve()
    found = None
    for parent in [cur] + list(cur.parents):
        if (parent / 'pyproject.toml').exists() or (parent / 'README.md').exists() or (parent / '.git').exists():
            found = parent
            break
    if found is not None:
        project_root = found
        break
print('Project root:', project_root)
pairs = [
    ('test', project_root / 'data' / 'processed' / 'llm_results_20260423_test.csv', project_root / 'data' / 'processed' / 'llm_results_test_2026-05-11_18-13-46.csv'),
    ('val', project_root / 'data' / 'processed' / 'llm_results_20260509_valid.csv', project_root / 'data' / 'processed' / 'llm_results_val_2026-05-10_23-21-44.csv'),
    ('train', project_root / 'data' / 'processed' / 'llm_results_20260511_train.csv', project_root / 'data' / 'processed' / 'llm_results_train_2026-05-08_23-34-10.csv'),
]
merged_dfs = {}
for split_name, a, b in pairs:
    df_a = safe_read_csv(a)
    df_b = safe_read_csv(b)
    if df_a.empty and df_b.empty:
        print(f'Both files missing or empty: {a}, {b}')
        continue
    merged = pd.concat([df_a, df_b], ignore_index=True, sort=False)
    merged_dfs[split_name] = merged
    print(f'{split_name}: merged rows={len(merged)} from {a.name} + {b.name}')

Project root: C:\Users\natal\Desktop\STUDIA MGR\SEM1\SIWY\LLM-Behavior-XAI
test: merged rows=600 from llm_results_20260423_test.csv + llm_results_test_2026-05-11_18-13-46.csv
val: merged rows=480 from llm_results_20260509_valid.csv + llm_results_val_2026-05-10_23-21-44.csv
train: merged rows=1920 from llm_results_20260511_train.csv + llm_results_train_2026-05-08_23-34-10.csv


In [9]:
def analyze_df(df: pd.DataFrame, name: str) -> Dict:
    report = {'name': name}
    prompt_id_col = find_column(df, ['prompt_id', 'promptid', 'prompt id', 'prompt_uuid', 'prompt_uuid_id'])
    prompt_col = find_column(df, ['prompt', 'instruction', 'input', 'query'])
    resp_col = find_column(df, ['response', 'answer', 'output', 'text'])
    lang_col = find_column(df, ['lang', 'language', 'locale'])
    paraphrase_col = find_column(df, ['paraphrase', 'is_paraphrase', 'has_paraphrase', 'with_paraphrase'])
    variant_col = find_column(df, ['variant', 'type', 'response_type', 'mode', 'form'])

    report['total_rows'] = len(df)

    # error detection
    err_cols = [c for c in df.columns if 'error' in c.lower()]
    if err_cols:
        is_error = df[err_cols].notna().any(axis=1)
        report['error_columns'] = err_cols
    else:
        if resp_col is not None:
            is_error = df[resp_col].isna() | (df[resp_col].astype(str).str.strip() == '')
        else:
            is_error = pd.Series([False] * len(df))
    report['error_count'] = int(is_error.sum())

    # prompt ids info
    if prompt_id_col is not None:
        report['unique_prompt_ids'] = int(df[prompt_id_col].nunique())
    elif prompt_col is not None:
        report['unique_prompt_ids'] = int(df[prompt_col].nunique())
    else:
        report['unique_prompt_ids'] = None

    # duplicates of responses per prompt_id
    prompt_key_col = prompt_id_col or prompt_col
    if prompt_key_col is not None and resp_col is not None:
        grp = df[[prompt_key_col, resp_col]].dropna()
        duplicate_mask = grp.duplicated(keep='first')
        report['duplicate_rows_same_prompt_response'] = int(duplicate_mask.sum())
        counts = grp.drop_duplicates().groupby(prompt_key_col)[resp_col].nunique()
        report['prompts_with_multiple_unique_responses'] = int((counts > 1).sum())
        report['unique_responses_per_prompt_stats'] = counts.describe().to_dict() if len(counts) else {}
    else:
        report['duplicate_rows_same_prompt_response'] = None
        report['prompts_with_multiple_unique_responses'] = None
        report['unique_responses_per_prompt_stats'] = None

    # required variants per prompt_id: PL/EN with and without paraphrase
    report['prompt_ids_with_full_variant_set'] = None
    report['missing_variants_by_prompt_id'] = []
    if prompt_key_col is not None and (lang_col is not None or paraphrase_col is not None or variant_col is not None):
        working = df.copy()

        def normalize_lang(x):
            s = str(x).lower()
            if s.startswith('pl') or 'pol' in s:
                return 'pl'
            if s.startswith('en') or 'eng' in s:
                return 'en'
            return None

        working['_lang_norm'] = working[lang_col].map(normalize_lang) if lang_col is not None else None

        if paraphrase_col is not None:
            def normalize_paraphrase(x):
                s = str(x).lower()
                if s in {'1', 'true', 'yes', 'y', 't'} or 'para' in s:
                    return 'paraphrase'
                return 'no_paraphrase'

            working['_para_norm'] = working[paraphrase_col].map(normalize_paraphrase)
        elif variant_col is not None:
            def normalize_paraphrase_from_variant(x):
                s = str(x).lower()
                if 'para' in s:
                    return 'paraphrase'
                return 'no_paraphrase'

            working['_para_norm'] = working[variant_col].map(normalize_paraphrase_from_variant)
        else:
            response_source = resp_col if resp_col is not None else prompt_col

            def normalize_paraphrase_from_text(x):
                s = str(x).lower()
                if 'para' in s:
                    return 'paraphrase'
                return 'no_paraphrase'

            working['_para_norm'] = working[response_source].map(normalize_paraphrase_from_text)

        working = working.dropna(subset=['_lang_norm'])
        working = working.drop_duplicates(subset=[prompt_key_col, '_lang_norm', '_para_norm'])

        expected = {
            ('pl', 'no_paraphrase'): 'PL without paraphrase',
            ('pl', 'paraphrase'): 'PL with paraphrase',
            ('en', 'no_paraphrase'): 'EN without paraphrase',
            ('en', 'paraphrase'): 'EN with paraphrase',
        }

        missing_rows = []
        full_count = 0
        for prompt_id, group in working.groupby(prompt_key_col):
            present = set(zip(group['_lang_norm'], group['_para_norm']))
            missing = [label for key, label in expected.items() if key not in present]
            if missing:
                missing_rows.append({'prompt_id': prompt_id, 'missing': missing})
            else:
                full_count += 1

        report['prompt_ids_with_full_variant_set'] = int(full_count)
        report['missing_variants_by_prompt_id'] = missing_rows
    else:
        report['prompt_ids_with_full_variant_set'] = None
        report['missing_variants_by_prompt_id'] = None

    return report

def deduplicate_by_prompt_and_response(df: pd.DataFrame) -> pd.DataFrame:
    prompt_col = find_column(df, ['prompt', 'instruction', 'input', 'query'])
    resp_col = find_column(df, ['response', 'answer', 'output', 'text'])
    if prompt_col is None or resp_col is None:
        return df.copy()
    dup_mask = df[[prompt_col, resp_col]].duplicated(keep='last')
    return df.loc[~dup_mask].copy()

def get_error_mask(df: pd.DataFrame) -> pd.Series:
    resp_col = find_column(df, ['response', 'answer', 'output', 'text'])
    err_cols = [c for c in df.columns if 'error' in c.lower()]
    if err_cols:
        return df[err_cols].notna().any(axis=1)
    if resp_col is not None:
        return df[resp_col].isna() | (df[resp_col].astype(str).str.strip() == '')
    return pd.Series([False] * len(df), index=df.index)

# Remove error rows, deduplicate, and save the three final files only
reports = {}
output_paths = {}
for name, df in merged_dfs.items():
    if df.empty:
        print(f'Skipping analysis for empty {name}')
        continue
    error_mask = get_error_mask(df)
    df_clean = df.loc[~error_mask].copy()
    removed_errors = int(error_mask.sum())
    deduped = deduplicate_by_prompt_and_response(df_clean)
    removed_duplicates = len(df_clean) - len(deduped)
    out_path = project_root / 'data' / 'processed' / f'llm_results_{name}_full.csv'
    deduped.to_csv(out_path, index=False)
    output_paths[name] = out_path
    print(f'{name}: removed error rows={removed_errors}, removed duplicate rows={removed_duplicates}, saved={out_path.name}, rows={len(deduped)}')
    r = analyze_df(deduped, name)
    reports[name] = r
    print(f'Report for {name}:')
    for k, v in r.items():
        if k == 'missing_variants_by_prompt_id' and isinstance(v, list):
            if not v:
                print('  missing_variants_by_prompt_id: []')
            else:
                print('  missing_variants_by_prompt_id:')
                for item in v:
                    print(f"    prompt_id={item['prompt_id']}: missing {', '.join(item['missing'])}")
            continue
        print(f'  {k}: {v}')
    print()

# Print summary dataframe
if reports:
    df_summary = pd.DataFrame.from_dict(reports, orient='index')
    print('Summary table:')
    print(df_summary.to_string())
    print('Saved files:')
    for name, path in output_paths.items():
        print(f'  {name}: {path}')


test: removed error rows=106, removed duplicate rows=0, saved=llm_results_test_full.csv, rows=494
Report for test:
  name: test
  total_rows: 494
  error_columns: ['error']
  error_count: 0
  unique_prompt_ids: 30
  duplicate_rows_same_prompt_response: 0
  prompts_with_multiple_unique_responses: 30
  unique_responses_per_prompt_stats: {'count': 30.0, 'mean': 16.466666666666665, 'std': 1.1665845619713493, 'min': 16.0, '25%': 16.0, '50%': 16.0, '75%': 16.0, 'max': 20.0}
  prompt_ids_with_full_variant_set: 30
  missing_variants_by_prompt_id: []

val: removed error rows=0, removed duplicate rows=0, saved=llm_results_val_full.csv, rows=480
Report for val:
  name: val
  total_rows: 480
  error_columns: ['error']
  error_count: 0
  unique_prompt_ids: 30
  duplicate_rows_same_prompt_response: 0
  prompts_with_multiple_unique_responses: 30
  unique_responses_per_prompt_stats: {'count': 30.0, 'mean': 16.0, 'std': 0.0, 'min': 16.0, '25%': 16.0, '50%': 16.0, '75%': 16.0, 'max': 16.0}
  prompt_ids_